In [9]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from utils import (
    get_df,
    compare_dataframes,
    write_parquet
)

In [10]:
df = get_df("./datasets/nhagar/fineweb_urls/")

In [11]:
con = duckdb.connect()
con.register("df", df)
print(df.head())

                                                 url                   domain
0  http://0ncemorewithfeeling.blogspot.com/2010/1...             blogspot.com
1  http://1001plus.blogspot.com/2011/11/very-shor...             blogspot.com
2  http://101bestandbrightest.com/companies/aviso...  101bestandbrightest.com
3  http://101bestandbrightest.com/companies/field...  101bestandbrightest.com
4    http://10pointsmanagement.com/the-wizard-of-oz/   10pointsmanagement.com


In [23]:
compress_df = con.execute("""
WITH domain_cleaned AS (
  SELECT
    url,
    domain,
    CASE
      WHEN RIGHT(domain, 1) = '.' THEN LEFT(domain, LENGTH(domain) - 1)
      ELSE domain
    END AS domain_stripped,
    RIGHT(domain, 1) = '.' AS has_dot
  FROM df
),

positions AS (
  SELECT
    url,
    domain,
    domain_stripped,
    POSITION(domain_stripped IN url) - 1 AS domain_offset,
    LENGTH(domain_stripped) AS domain_length,
    has_dot AS is_outlier
  FROM domain_cleaned
)

SELECT
  SUBSTR(url, 0, domain_offset + 1) AS url_prefix,
  SUBSTR(url, domain_offset + domain_length + 1) AS url_suffix,
  domain,
  is_outlier
FROM positions;
""").fetchdf()
print(compress_df.head)
con.register("compress_df", compress_df)

<bound method NDFrame.head of                          url_prefix  \
0       http://0ncemorewithfeeling.   
1                  http://1001plus.   
2                           http://   
3                           http://   
4                           http://   
...                             ...   
999995                      http://   
999996                      http://   
999997                      http://   
999998                      http://   
999999          http://keeganxusmf.   

                                               url_suffix  \
0                          /2010/11/shopping-at-home.html   
1                       /2011/11/very-short-marathon.html   
2                                    /companies/avisolve/   
3                                   /companies/fieldedge/   
4                                      /the-wizard-of-oz/   
...                                                   ...   
999995                                   /?p=4414&lang=en   
999996  /inde

In [27]:
restore_df = con.execute("""
SELECT
  url_prefix || 
  CASE
    WHEN is_outlier THEN LEFT(domain, LENGTH(domain) - 1)
    ELSE domain
  END || url_suffix AS url,
  domain
FROM compress_df
""").fetchdf()
print(restore_df.head)

<bound method NDFrame.head of                                                       url  \
0       http://0ncemorewithfeeling.blogspot.com/2010/1...   
1       http://1001plus.blogspot.com/2011/11/very-shor...   
2       http://101bestandbrightest.com/companies/aviso...   
3       http://101bestandbrightest.com/companies/field...   
4         http://10pointsmanagement.com/the-wizard-of-oz/   
...                                                   ...   
999995                 http://kasipkor.kz/?p=4414&lang=en   
999996  http://katm.co.uk/index.php/mother-stabs-boyfr...   
999997  http://kayser-immobiliendienst.de/en/component...   
999998                      http://kazworld.info/?p=64051   
999999  http://keeganxusmf.jaiblogs.com/2046180/not-kn...   

                            domain  
0                     blogspot.com  
1                     blogspot.com  
2          101bestandbrightest.com  
3          101bestandbrightest.com  
4           10pointsmanagement.com  
...            

In [28]:
compare_dataframes(df, restore_df)

DataFrames are equal


In [29]:
print(df.iloc[300999])
print(compress_df.iloc[300999])
print(restore_df.iloc[300999])

url       https://199.116.77.214/hair-transplant/post-su...
domain                                      199.116.77.214.
Name: 300999, dtype: object
url_prefix                              https://
url_suffix    /hair-transplant/post-surgical-gel
domain                           199.116.77.214.
is_outlier                                  True
Name: 300999, dtype: object
url       https://199.116.77.214/hair-transplant/post-su...
domain                                      199.116.77.214.
Name: 300999, dtype: object


In [30]:
write_parquet(compress_df, "./datasets_compress/nhagar/fineweb_urls/nhagar_fineweb_urls-2.parquet")